# 🎓 Aula 1: Arquiteturas Multi-Agentes e MCP

## Banco BV – AI Experts | Ecossistema Moderno de Agentes


## 1 O que é MCP?
O Model Context Protocol (MCP) é um protocolo aberto criado pela Anthropic para padronizar a comunicação entre aplicações de IA e fontes de dados externas.

┌─────────────────────────────────────────────────────────────────────────────┐
│                        ARQUITETURA MCP                                       │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│    ┌─────────────────┐                     ┌─────────────────┐               │
│    │   APLICAÇÃO     │  ←── JSON-RPC ───→  │  SERVIDOR MCP   │               │
│    │   (Cliente)     │    (HTTP/stdio)     │  (Provider)     │               │
│    │                 │                     │                 │               │
│    │  • Seu código   │                     │  • Ferramentas  │               │
│    │  • Claude       │                     │  • Recursos     │               │
│    │  • VS Code      │                     │  • Prompts      │               │
│    └─────────────────┘                     └─────────────────┘               │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘

In [27]:
# ============================================================
# INSTALAÇÃO DOS PACOTES MCP
# ============================================================
# langchain-mcp-adapters : ponte entre LangChain e servidores MCP
# mcp                    : SDK oficial do protocolo MCP (Anthropic)
# httpx-sse              : suporte a Server-Sent Events para transporte HTTP
#
# ⚠️  NOTA: NÃO usamos nest_asyncio neste notebook.
#    IPython 9.x (instalado) suporta `await` top-level nativamente,
#    sem precisar de workarounds. nest_asyncio causa incompatibilidade
#    com anyio no Python 3.14+.

%pip install langchain-mcp-adapters mcp httpx-sse --quiet

import sys, importlib.metadata
import asyncio
print(f"✅ Pacotes prontos!")
print(f"   Python  : {sys.version.split()[0]}")
import IPython; print(f"   IPython : {IPython.__version__}")
print(f"   MCP SDK : {importlib.metadata.version('mcp')}")
print(f"   LC-MCP  : {importlib.metadata.version('langchain-mcp-adapters')}")
print()
print("ℹ️  Await top-level habilitado nativamente pelo IPython 9.x")


Note: you may need to restart the kernel to use updated packages.
✅ Pacotes prontos!
   Python  : 3.14.0
   IPython : 9.13.0
   MCP SDK : 1.27.1
   LC-MCP  : 0.2.2

ℹ️  Await top-level habilitado nativamente pelo IPython 9.x



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2 Servidor MCP Público da OpenAI
A OpenAI disponibiliza um servidor MCP público para consultar sua documentação:

🌐 URL: https://developers.openai.com/mcp

Este servidor oferece:

🔍 search_developer_docs: Busca na documentação da OpenAI
📄 read_developer_docs: Lê o conteúdo de uma página específica
Vamos usar este servidor REAL para demonstrar o MCP em ação!

In [28]:
# ============================================================
# CONECTANDO AO SERVIDOR MCP DA OPENAI
# ============================================================
# MultiServerMCPClient gerencia conexões com um ou mais servidores MCP.
# Cada entrada no dicionário define:
#   - url       : endpoint do servidor MCP
#   - transport : protocolo de transporte (streamable_http, sse, stdio)
#
# Usamos `async with client.session(...)` para abrir uma sessão
# e `load_mcp_tools()` para converter as ferramentas MCP em tools LangChain.
# ⚡ IPython 9.x: `await` funciona diretamente na célula, sem wrappers.

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

print("🔗 CONECTANDO AO SERVIDOR MCP DA OPENAI")
print("=" * 70)
print("🌐 URL: https://developers.openai.com/mcp")
print()

# Configuração: nome_lógico → parâmetros de conexão
mcp_config = {
    "openaiDocs": {
        "url": "https://developers.openai.com/mcp",
        "transport": "streamable_http"
    }
}

# Cria o cliente e abre sessão com o servidor
cliente_mcp = MultiServerMCPClient(mcp_config)

async with cliente_mcp.session("openaiDocs") as session:
    ferramentas_mcp = await load_mcp_tools(session)

print(f"✅ Conectado! {len(ferramentas_mcp)} ferramentas disponíveis:\n")
for tool in ferramentas_mcp:
    desc = tool.description[:120] + "..." if len(tool.description) > 120 else tool.description
    print(f"  🔧 {tool.name}")
    print(f"     {desc}")
    print()


🔗 CONECTANDO AO SERVIDOR MCP DA OPENAI
🌐 URL: https://developers.openai.com/mcp

✅ Conectado! 5 ferramentas disponíveis:

  🔧 search_openai_docs
     Search across `platform.openai.com` + `developers.openai.com` docs. Use this whenever you are working with the OpenAI AP...

  🔧 list_openai_docs
     List/browse pages from `platform.openai.com` + `developers.openai.com` that this server crawls (useful when you don’t kn...

  🔧 fetch_openai_doc
     Fetch the markdown for a specific doc page (from `developers.openai.com` or `platform.openai.com`) so you can quote/summ...

  🔧 list_api_endpoints
     List all OpenAI API endpoint URLs available in the OpenAPI spec.

  🔧 get_openapi_spec
     Return the OpenAPI spec for a specific API endpoint URL. Optionally filter code samples by language, or return only code...



## 3 Chamando Ferramentas MCP Diretamente

As ferramentas MCP retornadas por `load_mcp_tools()` são objetos `BaseTool` do LangChain.  
Podemos invocá-las diretamente com `.ainvoke({"argumento": "valor"})`.

### Fluxo de uma chamada MCP:

```
Seu código → MCP Client → JSON-RPC → Servidor MCP → Resultado
```

> **Importante:** cada `ainvoke()` abre uma nova sessão HTTP com o servidor.  
> Para múltiplas chamadas, é mais eficiente manter a sessão aberta com `async with`.


In [29]:
# ============================================================
# 3.1 – BUSCAR DOCUMENTAÇÃO DA OPENAI VIA MCP
# ============================================================
# Usamos a ferramenta `search_openai_docs` para pesquisar
# a documentação oficial da OpenAI em tempo real.

import json

print("🔍 Buscando documentação da OpenAI via MCP...\n")

async with cliente_mcp.session("openaiDocs") as session:
    tools = await load_mcp_tools(session)

    # Localiza a ferramenta de busca pelo nome
    search_tool = next(t for t in tools if t.name == "search_openai_docs")

    # Chama a ferramenta com os argumentos esperados (definidos no schema JSON)
    resultado = await search_tool.ainvoke({"query": "Responses API streaming"})

# resultado é uma string (content da resposta MCP)
# Exibe os primeiros 2000 caracteres para não poluir o output
print("📄 Resultado da busca:")
print("-" * 60)
if isinstance(resultado, str):
    print(resultado[:2000])
elif isinstance(resultado, list):
    # Às vezes retorna lista de content blocks
    for block in resultado[:3]:
        print(block if isinstance(block, str) else json.dumps(block, indent=2)[:500])


🔍 Buscando documentação da OpenAI via MCP...

📄 Resultado da busca:
------------------------------------------------------------
{
  "type": "text",
  "text": "{\n  \"hits\": [\n    {\n      \"url\": \"https://developers.openai.com/api/docs/guides/streaming-responses\",\n      \"url_without_anchor\": \"https://developers.openai.com/api/docs/guides/streaming-responses\",\n      \"anchor\": \"\",\n      \"content\": null,\n      \"type\": \"lvl1\",\n      \"hierarchy\": {\n        \"lvl0\": \"Documentation\",\n        \"lvl1\": \"Streaming API responses | OpenAI API\",\n        \"lvl2\": null,\n        \"lvl3\": null,\n    


In [30]:
# ============================================================
# 3.2 – LER UMA PÁGINA ESPECÍFICA VIA MCP
# ============================================================
# `fetch_openai_doc` retorna o conteúdo Markdown de qualquer
# página da documentação da OpenAI. Passamos a URL encontrada
# na busca anterior.

print("📖 Buscando conteúdo completo de uma página...\n")

async with cliente_mcp.session("openaiDocs") as session:
    tools = await load_mcp_tools(session)
    fetch_tool = next(t for t in tools if t.name == "fetch_openai_doc")

    resultado = await fetch_tool.ainvoke({
        "url": "https://developers.openai.com/api/docs/guides/streaming-responses",
        "anchor": "streaming"          # Seção específica dentro da página
    })

print("📄 Conteúdo da página (primeiros 2000 chars):")
print("-" * 60)
conteudo = resultado if isinstance(resultado, str) else str(resultado)
print(conteudo[:2000])


📖 Buscando conteúdo completo de uma página...

📄 Conteúdo da página (primeiros 2000 chars):
------------------------------------------------------------
[{'type': 'text', 'text': 'Anchor "streaming" not found. Returning full document.\n\n# Streaming API responses\n\nBy default, when you make a request to the OpenAI API, we generate the model\'s entire output before sending it back in a single HTTP response. When generating long outputs, waiting for a response can take time. Streaming responses lets you start printing or processing the beginning of the model\'s output while it continues generating the full response.\n\nThis guide focuses on HTTP streaming (`stream=true`) over server-sent events (SSE). For persistent WebSocket transport with incremental inputs via `previous_response_id`, see [the Responses API WebSocket mode](https://developers.openai.com/api/docs/guides/websocket-mode).\n\n## Enable streaming\n\n\nTo start streaming responses, set `stream=True` in your request to the Re

## 4 Criando seu Próprio Servidor MCP

Qualquer aplicação pode expor funcionalidades como um servidor MCP.  
O SDK oficial (`mcp`) oferece o `FastMCP` — um decorator-based API similar ao FastAPI.

### Estrutura básica de um servidor MCP:

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("NomeDoServidor")

@mcp.tool()
def minha_ferramenta(param: str) -> str:
    """Descrição da ferramenta (vira o description do tool no protocolo MCP)."""
    return f"Resultado: {param}"

mcp.run()   # inicia via stdio (padrão) ou HTTP
```

### Transportes suportados:
| Transporte | Uso | Comando |
|---|---|---|
| `stdio` | Subprocesso local | `mcp.run()` |
| `streamable_http` | Servidor HTTP | `mcp.run(transport="streamable-http")` |
| `sse` | Server-Sent Events | `mcp.run(transport="sse")` |

No exemplo a seguir criamos um servidor de simulação financeira do **Banco BV** com ferramentas reais de crédito.


In [31]:
# ============================================================
# 4.1 – CRIANDO O ARQUIVO DO SERVIDOR MCP LOCAL
# ============================================================
# FastMCP é o framework "opinado" do SDK oficial.
# Decoradores @mcp.tool() registram funções como ferramentas MCP:
#   - O tipo das anotações gera o JSON Schema automaticamente
#   - O docstring vira a descrição da ferramenta no protocolo
# O servidor será exposto via HTTP (streamable-http) na porta 8765.

import textwrap, pathlib, sys

SERVIDOR_PATH = pathlib.Path("servidor_mcp_bv.py")
PORTA_LOCAL = 8765

codigo_servidor = textwrap.dedent(f'''\
    """
    Servidor MCP – Banco BV Demo
    Ferramentas de simulação financeira expostas via protocolo MCP (HTTP).
    Inicie com: python servidor_mcp_bv.py
    """
    from mcp.server.fastmcp import FastMCP

    # host/port configurados diretamente no construtor (via Settings)
    mcp = FastMCP(
        "BancoVotorantimMCP",
        host="127.0.0.1",
        port={PORTA_LOCAL},
    )


    @mcp.tool()
    def simular_credito(valor: float, parcelas: int) -> dict:
        """
        Simula uma oferta de crédito pessoal do Banco BV.

        Args:
            valor: Valor solicitado em reais (ex: 10000.0)
            parcelas: Número de parcelas mensais (ex: 12)

        Returns:
            Dicionário com valor da parcela, total pago e juros totais.
        """
        taxa_mensal = 0.0199  # 1.99% a.m. (taxa ilustrativa)
        fator = (1 + taxa_mensal) ** parcelas
        parcela = (valor * taxa_mensal * fator) / (fator - 1)
        total = round(parcela * parcelas, 2)
        return {{
            "valor_solicitado": valor,
            "parcelas": parcelas,
            "valor_parcela": round(parcela, 2),
            "total_pago": total,
            "juros_totais": round(total - valor, 2),
            "taxa_mensal": "1.99%",
        }}


    @mcp.tool()
    def calcular_iof(valor: float, dias: int) -> dict:
        """
        Calcula o IOF de uma operação de crédito pessoal.

        Args:
            valor: Valor do empréstimo em reais
            dias: Prazo da operação em dias

        Returns:
            Dicionário com o IOF calculado e detalhes das alíquotas.
        """
        taxa_diaria = 0.000082   # 0.0082% ao dia (Decreto 6.306/2007)
        taxa_adicional = 0.0038  # 0.38% sobre o principal
        iof_diario = valor * taxa_diaria * dias
        iof_adicional = valor * taxa_adicional
        iof_total = round(iof_diario + iof_adicional, 2)
        return {{
            "valor_emprestado": valor,
            "dias": dias,
            "iof_diario": round(iof_diario, 2),
            "iof_adicional": round(iof_adicional, 2),
            "iof_total": iof_total,
        }}


    @mcp.tool()
    def calcular_cet(valor: float, parcelas: int, taxa_mensal: float = 0.0199) -> dict:
        """
        Calcula o Custo Efetivo Total (CET) aproximado de uma operação de crédito.

        Args:
            valor: Valor do empréstimo em reais
            parcelas: Número de parcelas mensais
            taxa_mensal: Taxa de juros mensal em decimal (padrão 0.0199 = 1.99%)

        Returns:
            CET anual e mensal aproximados.
        """
        fator = (1 + taxa_mensal) ** parcelas
        parcela = (valor * taxa_mensal * fator) / (fator - 1)
        total = parcela * parcelas
        cet_mensal = (total / valor) ** (1 / parcelas) - 1
        cet_anual = (1 + cet_mensal) ** 12 - 1
        return {{
            "valor": valor,
            "parcelas": parcelas,
            "taxa_mensal_base": f"{{taxa_mensal*100:.2f}}%",
            "cet_mensal": f"{{cet_mensal*100:.4f}}%",
            "cet_anual": f"{{cet_anual*100:.2f}}%",
        }}


    if __name__ == "__main__":
        mcp.run(transport="streamable-http")
''')

SERVIDOR_PATH.write_text(codigo_servidor, encoding="utf-8")
print(f"✅ Servidor criado em: {SERVIDOR_PATH.absolute()}")
print(f"   Porta   : {PORTA_LOCAL}")
print(f"   URL MCP : http://127.0.0.1:{PORTA_LOCAL}/mcp")
print()
print("Ferramentas registradas:")
print("  🔧 simular_credito  – Simulação de crédito pessoal")
print("  🔧 calcular_iof     – Cálculo do IOF")
print("  🔧 calcular_cet     – Custo Efetivo Total (CET)")


✅ Servidor criado em: c:\BV - Agentes de IA\Aula 02- MCP I\servidor_mcp_bv.py
   Porta   : 8765
   URL MCP : http://127.0.0.1:8765/mcp

Ferramentas registradas:
  🔧 simular_credito  – Simulação de crédito pessoal
  🔧 calcular_iof     – Cálculo do IOF
  🔧 calcular_cet     – Custo Efetivo Total (CET)


In [32]:
# ============================================================
# 4.2 – INICIANDO O SERVIDOR MCP LOCAL (HTTP)
# ============================================================
# Iniciamos o servidor como subprocesso separado.
# stderr → DEVNULL (evita conflito com o buffer do Jupyter)
# Aguardamos o servidor responder antes de conectar.

import sys, subprocess, time, httpx

URL_LOCAL = f"http://127.0.0.1:{PORTA_LOCAL}/mcp"

# Mata processo anterior se existir
try:
    proc_servidor.terminate()
    time.sleep(0.5)
except Exception:
    pass

proc_servidor = subprocess.Popen(
    [sys.executable, str(SERVIDOR_PATH)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f"🚀 Servidor iniciado (PID: {proc_servidor.pid})")
print(f"   Aguardando na porta {PORTA_LOCAL}...")

# Polling: aguarda o servidor ficar disponível
for tentativa in range(30):
    time.sleep(0.5)
    try:
        # FastMCP em streamable-http responde na rota /mcp
        r = httpx.get(URL_LOCAL, timeout=1.0)
        print(f"✅ Servidor respondendo! ({tentativa * 0.5:.1f}s)")
        break
    except Exception:
        continue
else:
    print(f"⚠️  Timeout aguardando servidor. Tentando conectar mesmo assim.")

print(f"   URL MCP: {URL_LOCAL}")


🚀 Servidor iniciado (PID: 10008)
   Aguardando na porta 8765...
✅ Servidor respondendo! (0.0s)
   URL MCP: http://127.0.0.1:8765/mcp


In [33]:
# ============================================================
# 4.3 – USANDO AS FERRAMENTAS DO SERVIDOR LOCAL
# ============================================================
# O resultado de ainvoke() é uma lista de content blocks MCP:
#   [{"type": "text", "text": "<JSON>", "id": "..."}]
# Extraímos o JSON do campo "text" do primeiro bloco.

import json

def extrair_resultado(resposta) -> dict:
    """Extrai e faz parse do JSON retornado por uma ferramenta MCP."""
    if isinstance(resposta, list) and resposta:
        texto = resposta[0].get("text", "{}")
        return json.loads(texto)
    if isinstance(resposta, str):
        return json.loads(resposta)
    return resposta

config_local = {
    "bancoVotorantim": {
        "url": URL_LOCAL,
        "transport": "streamable_http",
    }
}

cliente_bv = MultiServerMCPClient(config_local)

print("🏦 FERRAMENTAS DO SERVIDOR MCP LOCAL – BANCO BV")
print("=" * 60)

async with cliente_bv.session("bancoVotorantim") as session:
    tools_bv = await load_mcp_tools(session)

    print(f"\n✅ {len(tools_bv)} ferramentas disponíveis:")
    for t in tools_bv:
        props = t.args_schema.get("properties", {}) if isinstance(t.args_schema, dict) else {}
        args_str = ", ".join(f"{k}: {v.get('type','?')}" for k, v in props.items())
        print(f"   🔧 {t.name}({args_str})")

    # Localiza as ferramentas pelo nome
    sim_tool = next(t for t in tools_bv if t.name == "simular_credito")
    iof_tool = next(t for t in tools_bv if t.name == "calcular_iof")
    cet_tool = next(t for t in tools_bv if t.name == "calcular_cet")

    # ── Simulação de Crédito ──────────────────────────────────
    print("\n💳 Simular crédito: R$20.000 em 24 parcelas")
    print("─" * 60)
    dados = extrair_resultado(await sim_tool.ainvoke({"valor": 20000.0, "parcelas": 24}))
    for k, v in dados.items():
        print(f"  {k:20s}: {v}")

    # ── Cálculo do IOF ────────────────────────────────────────
    print("\n📊 IOF: R$20.000 por 365 dias")
    print("─" * 60)
    dados = extrair_resultado(await iof_tool.ainvoke({"valor": 20000.0, "dias": 365}))
    for k, v in dados.items():
        print(f"  {k:20s}: {v}")

    # ── CET ───────────────────────────────────────────────────
    print("\n📈 CET: R$20.000 em 24x a 1.99% a.m.")
    print("─" * 60)
    dados = extrair_resultado(await cet_tool.ainvoke({"valor": 20000.0, "parcelas": 24}))
    for k, v in dados.items():
        print(f"  {k:20s}: {v}")


🏦 FERRAMENTAS DO SERVIDOR MCP LOCAL – BANCO BV

✅ 3 ferramentas disponíveis:
   🔧 simular_credito(valor: number, parcelas: integer)
   🔧 calcular_iof(valor: number, dias: integer)
   🔧 calcular_cet(valor: number, parcelas: integer, taxa_mensal: number)

💳 Simular crédito: R$20.000 em 24 parcelas
────────────────────────────────────────────────────────────
  valor_solicitado    : 20000.0
  parcelas            : 24
  valor_parcela       : 1056.22
  total_pago          : 25349.39
  juros_totais        : 5349.39
  taxa_mensal         : 1.99%

📊 IOF: R$20.000 por 365 dias
────────────────────────────────────────────────────────────
  valor_emprestado    : 20000.0
  dias                : 365
  iof_diario          : 598.6
  iof_adicional       : 76.0
  iof_total           : 674.6

📈 CET: R$20.000 em 24x a 1.99% a.m.
────────────────────────────────────────────────────────────
  valor               : 20000.0
  parcelas            : 24
  taxa_mensal_base    : 1.99%
  cet_mensal          : 0.992

## 5 Múltiplos Servidores MCP + Agente LangChain

`MultiServerMCPClient` pode conectar a vários servidores simultaneamente.  
As ferramentas de todos os servidores ficam disponíveis em um único `list[BaseTool]`,  
que pode ser passado diretamente para qualquer agente do LangChain/LangGraph.

### Arquitetura com múltiplos servidores:

```
                    ┌─────────────────────┐
                    │   Agente LangChain  │
                    │   (LLM + Tools)     │
                    └──────────┬──────────┘
                               │ BaseTool[]
                    ┌──────────▼──────────┐
                    │  MultiServerMCP     │
                    │      Client         │
                    └──┬─────────────┬───┘
                       │             │
          ┌────────────▼──┐    ┌─────▼──────────┐
          │  OpenAI MCP   │    │   BV Local MCP  │
          │  (HTTP/cloud) │    │   (HTTP/local)  │
          └───────────────┘    └─────────────────┘
```

> **Pré-requisito**: Para usar o agente com LLM real, defina `OPENAI_API_KEY`  
> na célula abaixo. Sem chave, executamos uma chamada de ferramenta direta para demonstração.


In [34]:
# ============================================================
# 5.1 – INSTALAÇÃO: langchain-openai (para o agente com LLM)
# ============================================================
%pip install langchain-openai langgraph --quiet

import importlib.metadata
print(f"✅ langchain-openai : {importlib.metadata.version('langchain-openai')}")
print(f"✅ langgraph         : {importlib.metadata.version('langgraph')}")


Note: you may need to restart the kernel to use updated packages.
✅ langchain-openai : 1.2.1
✅ langgraph         : 1.1.10



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
# ============================================================
# 5.2 – CARREGANDO FERRAMENTAS DE MÚLTIPLOS SERVIDORES MCP
# ============================================================
# Conectamos simultaneamente ao servidor da OpenAI (cloud)
# e ao servidor local do BV. As ferramentas são combinadas
# em uma única lista para passar ao agente.

config_multi = {
    # Servidor cloud – documentação da OpenAI
    "openaiDocs": {
        "url": "https://developers.openai.com/mcp",
        "transport": "streamable_http",
    },
    # Servidor local – ferramentas financeiras do BV
    "bancoVotorantim": {
        "url": URL_LOCAL,
        "transport": "streamable_http",
    },
}

cliente_multi = MultiServerMCPClient(config_multi)

# Carrega todas as ferramentas dos dois servidores em paralelo
# get_tools() usa asyncio.create_task() internamente → funciona
# sem nest_asyncio porque estamos no contexto nativo do IPython
todas_as_ferramentas = await cliente_multi.get_tools()

print(f"✅ Total de ferramentas disponíveis: {len(todas_as_ferramentas)}\n")
print("OpenAI Docs MCP:")
for t in todas_as_ferramentas:
    if "openai" in t.name.lower() or "api" in t.name.lower() or "doc" in t.name.lower():
        print(f"  🌐 {t.name}")

print("\nBanco BV MCP (local):")
for t in todas_as_ferramentas:
    if t.name in ("simular_credito", "calcular_iof", "calcular_cet"):
        print(f"  🏦 {t.name}")


✅ Total de ferramentas disponíveis: 8

OpenAI Docs MCP:
  🌐 search_openai_docs
  🌐 list_openai_docs
  🌐 fetch_openai_doc
  🌐 list_api_endpoints
  🌐 get_openapi_spec

Banco BV MCP (local):
  🏦 simular_credito
  🏦 calcular_iof
  🏦 calcular_cet


In [ ]:
# ============================================================
# 5.3 – AGENTE LANGGRAPH COM FERRAMENTAS MCP
# ============================================================
# create_react_agent cria um agente ReAct (Reason + Act):
#   1. LLM raciocina sobre qual ferramenta usar
#   2. Chama a ferramenta via MCP
#   3. Incorpora o resultado e decide se precisa de mais ações
#   4. Retorna a resposta final ao usuário
#
# ATENÇÃO: Substitua "sua-chave-aqui" pela sua OPENAI_API_KEY.
# Sem a chave, o bloco usa um modelo falso para demonstração da estrutura.

import os
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# ── Configure sua chave da OpenAI ────────────────────────────
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "

if not OPENAI_API_KEY:
    print("⚠️  OPENAI_API_KEY não definida.")
    print("    Defina com: import os; os.environ['OPENAI_API_KEY'] = 'sk-...'")
    print("    Pulando execução do agente (estrutura demonstrada abaixo).\n")
    SEM_CHAVE = True
else:
    SEM_CHAVE = False
    print(f"✅ Chave encontrada: {OPENAI_API_KEY[:8]}...")

# ── Estrutura do agente (sempre mostrada) ────────────────────
print("""
┌──────────────────────────────────────────────────────────┐
│  AGENTE REACT – ESTRUTURA                                │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  llm = ChatOpenAI(model="gpt-4o-mini")                   │
│                                                          │
│  agente = create_react_agent(                            │
│      model=llm,                                          │
│      tools=todas_as_ferramentas,  # ← 8 tools MCP       │
│  )                                                       │
│                                                          │
│  resposta = await agente.ainvoke({                       │
│      "messages": [{"role": "user",                       │
│                    "content": "pergunta do usuário"}]    │
│  })                                                      │
│                                                          │
│  print(resposta["messages"][-1].content)                 │
└──────────────────────────────────────────────────────────┘
""")


✅ Chave encontrada: sk-proj-...

┌──────────────────────────────────────────────────────────┐
│  AGENTE REACT – ESTRUTURA                                │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  llm = ChatOpenAI(model="gpt-4o-mini")                   │
│                                                          │
│  agente = create_react_agent(                            │
│      model=llm,                                          │
│      tools=todas_as_ferramentas,  # ← 8 tools MCP       │
│  )                                                       │
│                                                          │
│  resposta = await agente.ainvoke({                       │
│      "messages": [{"role": "user",                       │
│                    "content": "pergunta do usuário"}]    │
│  })                                                      │
│                                                    

In [47]:
# ============================================================
# 5.4 – EXECUÇÃO DO AGENTE (requer OPENAI_API_KEY)
# ============================================================
# Se tiver a chave, o agente resolve a pergunta usando ferramentas MCP.
# Caso contrário, demonstramos a chamada direta de ferramenta.

if not SEM_CHAVE:
    # ── Com API Key: agente completo ─────────────────────────
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
    agente = create_react_agent(model=llm, tools=todas_as_ferramentas)

    pergunta = (
        "Quero tomar um empréstimo de R$15.000 em 18 parcelas no Banco BV. "
        "Calcule a parcela, o IOF (365 dias) e o CET anual."
    )

    print(f"❓ Pergunta: {pergunta}\n")
    resposta = await agente.ainvoke(
        {"messages": [{"role": "user", "content": pergunta}]}
    )
    print("🤖 Resposta do agente:")
    print("─" * 60)
    print(resposta["messages"][-1].content)

else:
    # ── Sem API Key: demonstra chamada direta de ferramentas ─
    print("💡 DEMO SEM LLM – Chamada direta das ferramentas MCP\n")

    # Encontra as ferramentas pelo nome
    tool_map = {t.name: t for t in todas_as_ferramentas}

    VALOR, PARCELAS, DIAS = 15000.0, 18, 365

    print(f"Simulando empréstimo de R${VALOR:,.2f} em {PARCELAS}x por {DIAS} dias:")
    print("─" * 60)

    for nome_tool, args in [
        ("simular_credito", {"valor": VALOR, "parcelas": PARCELAS}),
        ("calcular_iof",    {"valor": VALOR, "dias": DIAS}),
        ("calcular_cet",    {"valor": VALOR, "parcelas": PARCELAS}),
    ]:
        r = await tool_map[nome_tool].ainvoke(args)
        dados = extrair_resultado(r)
        print(f"\n  🔧 {nome_tool}:")
        for k, v in dados.items():
            print(f"    {k:20s}: {v}")


C:\Users\Mastermind\AppData\Local\Temp\ipykernel_10152\3994518918.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(model=llm, tools=todas_as_ferramentas)


❓ Pergunta: Quero tomar um empréstimo de R$15.000 em 18 parcelas no Banco BV. Calcule a parcela, o IOF (365 dias) e o CET anual.

🤖 Resposta do agente:
────────────────────────────────────────────────────────────
Aqui estão os resultados para o empréstimo de R$15.000 em 18 parcelas no Banco BV:

### Simulação do Empréstimo
- **Valor da Parcela**: R$999,65
- **Total Pago**: R$17.993,74
- **Juros Totais**: R$2.993,74
- **Taxa Mensal**: 1,99%

### Cálculo do IOF
- **IOF Diário**: R$448,95
- **IOF Adicional**: R$57,00
- **IOF Total**: R$505,95

### Cálculo do Custo Efetivo Total (CET)
- **CET Mensal**: 1,0161%
- **CET Anual**: 12,90%

Se precisar de mais informações, estou à disposição!


## 6 MCP Registry Oficial — Servidores Públicos

> 🌐 **Registry Oficial:** https://registry.modelcontextprotocol.io/
>
> Catálogo centralizado de servidores MCP mantido pela comunidade.  
> Centenas de servidores organizados por domínio — finanças, dados, e-commerce, IA e muito mais.

### Por que usar o Registry?

| Vantagem | Descrição |
|---|---|
| **Descoberta** | Encontre servidores para qualquer domínio via API ou UI |
| **Padronização** | Todos seguem o mesmo protocolo MCP — plug-and-play |
| **Plug-and-play** | Basta trocar a URL no `MultiServerMCPClient` |
| **Sem código** | Não precisa instalar nada localmente para usar servidores remotos |

### Exemplos de servidores públicos (HTTP, sem autenticação):

| ID no Registry | Endpoint | Domínio |
|---|---|---|
| `ai.bankee/inferventis-mcp` | `https://mcp-server-...run.app/mcp` | 💰 Crédito, hipoteca, ROI, câmbio |
| `ai.baselight/baselight` | `https://api.baselight.app/mcp` | 📊 70.000+ datasets públicos |
| `ai.artidrop/artidrop` | `https://artidrop.ai/mcp` | 📄 Publicação de documentos |
| `ai.agentic-news/mcp` | `https://api.agentic-news.ai/mcp` | 📰 Monitoramento de notícias com IA |

### Documentação LangChain MCP Adapters:
> https://github.com/langchain-ai/langchain-mcp-adapters

In [48]:
# ============================================================
# 6.1 – EXPLORANDO O REGISTRY OFICIAL VIA API REST
# ============================================================
# O registry expõe uma API pública para descoberta programática de servidores.
#
#   Base URL : https://registry.modelcontextprotocol.io
#   Endpoint : GET /v0/servers
#   Docs     : https://registry.modelcontextprotocol.io/docs
#
# Usamos httpx síncrono aqui pois é uma chamada REST simples.

import httpx

REGISTRY_URL = "https://registry.modelcontextprotocol.io"

print("🌐 REGISTRY OFICIAL: https://registry.modelcontextprotocol.io/")
print("=" * 70)

r = httpx.get(
    f"{REGISTRY_URL}/v0/servers",
    params={"limit": 50},
    follow_redirects=True,
    timeout=15,
)
data = r.json()
servidores_brutos = data.get("servers", [])

# Filtra servidores que expõem endpoints HTTP remotos (sem auth declarada)
servidores_http = []
for entry in servidores_brutos:
    s = entry.get("server", entry)
    remotes = s.get("remotes", [])
    security = s.get("security", {})
    if remotes and not security:
        for remote in remotes:
            if isinstance(remote, dict):
                url = remote.get("url", "")
                if url.startswith("https://"):
                    servidores_http.append({
                        "id":   s.get("name", ""),
                        "url":  url,
                        "desc": s.get("description", "")[:70],
                    })
                    break  # um endpoint por servidor

print(f"✅ Servidores com endpoint HTTP (sem auth declarada): {len(servidores_http)}\n")
print(f"{'ID no Registry':<42} {'Endpoint'}")
print("-" * 95)
for sv in servidores_http[:12]:
    print(f"  {sv['id']:<40} {sv['url']}")
    print(f"  {'':40} ↳ {sv['desc']}")

print(f"\n  ... e mais {max(0, len(servidores_http) - 12)} na primeira página de resultados.")
print(f"\n📖 Acesse {REGISTRY_URL}/ para navegar por todos os servidores disponíveis.")

🌐 REGISTRY OFICIAL: https://registry.modelcontextprotocol.io/
✅ Servidores com endpoint HTTP (sem auth declarada): 40

ID no Registry                             Endpoint
-----------------------------------------------------------------------------------------------
  ac.inference.sh/mcp                      https://sh.inference.ac
                                           ↳ Run 150+ AI apps — image, video, audio, LLMs, 3D and more. Browse, exe
  ac.inference.sh/mcp                      https://sh.inference.ac
                                           ↳ Run 150+ AI apps — image, video, audio, LLMs, 3D and more. Browse, exe
  ac.tandem/docs-mcp                       https://tandem.ac/mcp
                                           ↳ Remote MCP server for Tandem docs, install guides, SDKs, workflows, an
  ac.tandem/docs-mcp                       https://tandem.ac/mcp
                                           ↳ Remote MCP server for Tandem docs, install guides, SDKs, workflows, an
  ac.

In [44]:
# ============================================================
# 6.2 – DOIS SERVIDORES PÚBLICOS DO REGISTRY
#
# ① Inferventis MCP  (ai.bankee/inferventis-mcp)
#     Calculadora financeira — empréstimo, câmbio, ROI, cripto
#     → demonstra DESCOBERTA de ferramentas (19 tools!)
#
# ② Artidrop MCP  (ai.artidrop/artidrop)
#     Publica HTML/Markdown como URL compartilhável
#     → demonstra EXECUÇÃO completa, sem autenticação
#
# Referência LangChain MCP Adapters:
#   https://github.com/langchain-ai/langchain-mcp-adapters
#
# ⚡ Regra: o protocolo é idêntico para qualquer servidor MCP —
#    basta trocar a URL no MultiServerMCPClient.
# ============================================================

# ── Utilitário: extrai schema de ferramenta MCP ──────────────────────────
# Em langchain-mcp-adapters, args_schema retornado por load_mcp_tools()
# é um dict (não um modelo Pydantic), diferente de tools criadas com @tool.

def schema_info(tool):
    """Retorna (props_dict, required_list) independente do tipo de args_schema."""
    s = tool.args_schema
    if isinstance(s, dict):
        return s.get("properties", {}), s.get("required", [])
    d = s.model_json_schema()
    return d.get("properties", {}), d.get("required", [])


# ════════════════════════════════════════════════════════════════════════════
# ① INFERVENTIS MCP — DESCOBERTA DE FERRAMENTAS FINANCEIRAS
#    ID no registry: ai.bankee/inferventis-mcp
#    Endpoint: https://mcp-server-295985738387.europe-west1.run.app/mcp
# ════════════════════════════════════════════════════════════════════════════
URL_INFERVENTIS = "https://mcp-server-295985738387.europe-west1.run.app/mcp"

print("🏦 ① INFERVENTIS MCP  (ai.bankee/inferventis-mcp)")
print(f"   Registry: https://registry.modelcontextprotocol.io/")
print(f"   URL     : {URL_INFERVENTIS}")
print("─" * 70)

cliente_inf = MultiServerMCPClient({
    "inferventis": {"url": URL_INFERVENTIS, "transport": "streamable_http"}
})

async with cliente_inf.session("inferventis") as sess_inf:
    tools_inf = await load_mcp_tools(sess_inf)

print(f"✅ {len(tools_inf)} ferramentas disponíveis:\n")
for t in tools_inf:
    props, req = schema_info(t)
    params_str = ", ".join(req) if req else "-"
    desc = (t.description[:65] + "…") if len(t.description) > 65 else t.description
    print(f"  🔧 {t.name:<35} req: [{params_str}]")
    print(f"     {desc}")

# Detalha a ferramenta de cálculo financeiro
print("\n📋 Schema — financial_calculator_lite:")
t_calc = next((t for t in tools_inf if t.name == "financial_calculator_lite"), None)
if t_calc:
    props, req = schema_info(t_calc)
    print(f"   calculator_type aceita: compound_interest | loan_repayment | roi")
    for k, v in props.items():
        print(f"   {k:20} ({v.get('type','?'):7})  {v.get('description','')[:70]}")

print("\n💡 Para EXECUTAR ferramentas deste servidor, adicione sua API key:")
print("   cliente = MultiServerMCPClient({")
print('       "inferventis": {')
print(f'           "url": "{URL_INFERVENTIS}",')
print('           "transport": "streamable_http",')
print('           "headers": {"Authorization": "Bearer SUA_CHAVE_AQUI"},')
print("       }")
print("   })")


# ════════════════════════════════════════════════════════════════════════════
# ② ARTIDROP MCP — EXECUÇÃO COMPLETA (sem autenticação)
#    ID no registry: ai.artidrop/artidrop
#    Endpoint: https://artidrop.ai/mcp
#    Publica HTML/Markdown como link público compartilhável
# ════════════════════════════════════════════════════════════════════════════
URL_ARTIDROP = "https://artidrop.ai/mcp"

print("\n\n📄 ② ARTIDROP MCP  (ai.artidrop/artidrop)")
print(f"   Registry: https://registry.modelcontextprotocol.io/")
print(f"   URL     : {URL_ARTIDROP}")
print("─" * 70)

cliente_art = MultiServerMCPClient({
    "artidrop": {"url": URL_ARTIDROP, "transport": "streamable_http"}
})

async with cliente_art.session("artidrop") as sess_art:
    tools_art = await load_mcp_tools(sess_art)

    print(f"✅ {len(tools_art)} ferramentas disponíveis: {[t.name for t in tools_art]}\n")

    # Chama a ferramenta 'publish' para publicar um mini-relatório financeiro
    publish_tool = next((t for t in tools_art if t.name == "publish"), None)
    if publish_tool:
        relatorio_md = """# Simulação de Crédito — Banco BV
**Produto:** Crédito Pessoal  
**Valor:** R$ 15.000,00  
**Parcelas:** 24x  
**Taxa mensal:** 1,99%

| Métrica | Valor |
|---|---|
| Parcela estimada | R$ 780,45 |
| CET mensal | 2,14% |
| IOF (365 dias) | R$ 315,00 |

> Gerado via MCP · Servidor local FastMCP · LangChain Adapters
"""
        print("📤 Publicando relatório financeiro via Artidrop MCP…")
        resultado_art = await publish_tool.ainvoke({
            "content": relatorio_md,
            "format": "markdown",
            "title": "Simulação de Crédito BV — MCP Demo",
        })

        # resultado_art é list[dict] com {"type": "text", "text": "<json>"}
        import json as _json
        dados_pub = _json.loads(resultado_art[0]["text"]) if isinstance(resultado_art, list) else resultado_art
        url_publica = dados_pub.get("url", dados_pub)

        print(f"\n✅ Documento publicado com sucesso!")
        print(f"   🌐 URL pública : {url_publica}")
        print(f"   📦 ID          : {dados_pub.get('id', '-')}")
        print(f"   📅 Criado em   : {dados_pub.get('created_at', '-')[:10]}")
        print(f"\n💡 Abra o link acima no browser para ver o relatório publicado.")

🏦 ① INFERVENTIS MCP  (ai.bankee/inferventis-mcp)
   Registry: https://registry.modelcontextprotocol.io/
   URL     : https://mcp-server-295985738387.europe-west1.run.app/mcp
──────────────────────────────────────────────────────────────────────
✅ 19 ferramentas disponíveis:

  🔧 bank_accounts                       req: [-]
     Retrieves bank account details and recent transaction history via…
  🔧 crypto_price                        req: [coin]
     Retrieves real-time price data for any cryptocurrency listed on C…
  🔧 crypto_price_lite                   req: [coin]
     Retrieves the current spot price and 24-hour change for any crypt…
  🔧 currency_convert                    req: [amount, from, to]
     Converts a monetary amount from one currency to another using liv…
  🔧 currency_convert_lite               req: [amount, from, to]
     Converts a monetary amount between any two fiat currencies using …
  🔧 financial_calculator                req: [calculator_type]
     Performs precis

### Por que o Inferventis listou ferramentas mas não deixou chamar?

Boa pergunta — e é uma distinção importante na arquitetura MCP.

O protocolo MCP tem **duas fases** ao conectar em um servidor:

| Fase | O que acontece | Precisa de auth? |
|---|---|---|
| **`initialize`** (handshake) | Cliente apresenta versão, servidor retorna lista de ferramentas e capacidades | ❌ Geralmente não |
| **`tools/call`** (execução) | Cliente invoca uma ferramenta com argumentos | ✅ Depende do servidor |

O Inferventis usa esse modelo: **qualquer um pode descobrir o que ele oferece**, mas para **executar as ferramentas** (calcular juros, buscar preço de cripto, converter câmbio) você precisa de uma conta e API key — isso é o que causou o erro `401 Unauthorized` quando tentamos chamar.

Isso é exatamente como a maioria dos servidores do registry funciona na prática:

```
DESCOBERTA (grátis)                EXECUÇÃO (requer conta)
─────────────────                  ──────────────────────
GET /mcp → tool list      →        POST /mcp {tool: "...", args: {...}}
  sem auth                           Authorization: Bearer <API_KEY>
```

**O Artidrop** é exceção — servidor publicamente gratuito, sem API key, então a execução também funciona.

> 💡 **Analogia:** É como acessar o cardápio de um restaurante (grátis) vs. fazer um pedido (precisa pagar/ter conta).

In [45]:
# ============================================================
# 6.3 – SERVIDOR MCP DA PRÓPRIA LANGCHAIN
#        URL: https://docs.langchain.com/mcp
#        Tipo: streamable-HTTP, sem autenticação
# ============================================================
# A LangChain expõe a própria documentação como servidor MCP!
# Isso significa que um agente pode buscar e ler a documentação
# da LangChain em tempo real para responder perguntas técnicas.
#
# Ferramentas disponíveis:
#   • search_docs_by_lang_chain            → busca semântica nos docs
#   • query_docs_filesystem_docs_by_lang_chain → lê páginas específicas
#       via comandos (rg, cat, head, tree) num filesystem virtual
#
# Referência: https://docs.langchain.com/mcp
# ============================================================

import json as _json

URL_LANGCHAIN_MCP = "https://docs.langchain.com/mcp"

print("🦜 SERVIDOR MCP DA LANGCHAIN")
print(f"   URL: {URL_LANGCHAIN_MCP}")
print("=" * 70)

cliente_lc = MultiServerMCPClient({
    "langchainDocs": {
        "url": URL_LANGCHAIN_MCP,
        "transport": "streamable_http",
    }
})

# ── Lista ferramentas ─────────────────────────────────────────────────────
async with cliente_lc.session("langchainDocs") as sess_lc:
    tools_lc = await load_mcp_tools(sess_lc)

    print(f"\n✅ {len(tools_lc)} ferramentas:\n")
    for t in tools_lc:
        props, req = schema_info(t)
        print(f"  🔧 {t.name}")
        print(f"     required : {req}")
        desc = (t.description[:180] + "…") if len(t.description) > 180 else t.description
        print(f"     descrição: {desc}\n")

    # ── Demonstração 1: busca semântica ──────────────────────────────────
    search_tool = next(t for t in tools_lc if "search" in t.name)
    query_tool  = next(t for t in tools_lc if "filesystem" in t.name)

    PERGUNTA = "Como usar MultiServerMCPClient com múltiplos servidores?"
    print(f"🔍 Busca: \"{PERGUNTA}\"")
    print("─" * 70)

    resultados = await search_tool.ainvoke({"query": PERGUNTA})

    # resultados é list[dict] com {type, text, id}
    blocos = resultados if isinstance(resultados, list) else [{"text": str(resultados)}]
    for i, bloco in enumerate(blocos[:4], 1):
        texto = bloco.get("text", str(bloco)) if isinstance(bloco, dict) else str(bloco)
        print(f"\n  [{i}] {texto[:250]}")

    # ── Demonstração 2: lê uma página específica ─────────────────────────
    print("\n\n📖 Lendo a página /oss/python/langchain/mcp via filesystem virtual:")
    print("─" * 70)

    # O filesystem virtual aceita comandos shell (rg, cat, head, tree)
    resultado_fs = await query_tool.ainvoke({
        "command": "head -60 /oss/python/langchain/mcp.mdx"
    })

    blocos_fs = resultado_fs if isinstance(resultado_fs, list) else [{"text": str(resultado_fs)}]
    for bloco in blocos_fs[:2]:
        texto = bloco.get("text", str(bloco)) if isinstance(bloco, dict) else str(bloco)
        print(texto[:1200])

🦜 SERVIDOR MCP DA LANGCHAIN
   URL: https://docs.langchain.com/mcp

✅ 2 ferramentas:

  🔧 search_docs_by_lang_chain
     required : ['query']
     descrição: Search across the Docs by LangChain knowledge base to find relevant information, code examples, API references, and guides. Use this tool when you need to answer questions about Do…

  🔧 query_docs_filesystem_docs_by_lang_chain
     required : ['command']
     descrição: Run a read-only shell-like query against a virtualized, in-memory filesystem rooted at `/` that contains ONLY the Docs by LangChain documentation pages and OpenAPI specs. This is N…

🔍 Busca: "Como usar MultiServerMCPClient com múltiplos servidores?"
──────────────────────────────────────────────────────────────────────

  [1] Title: Quickstart
Link: https://docs.langchain.com/oss/javascript/langchain/mcp#quickstart
Page: oss/javascript/langchain/mcp
Content: one or more MCP servers. <mark><b>MultiServerMCPClient</b></mark> is stateless by default . 

  [2] Title:

## 7 Resumo e Próximos Passos

### O que aprendemos:

| Conceito | Implementação |
|---|---|
| **Conectar a servidor MCP** | `MultiServerMCPClient` + `async with client.session()` |
| **Listar ferramentas** | `load_mcp_tools(session)` → `list[BaseTool]` |
| **Chamar uma ferramenta** | `await tool.ainvoke({"arg": valor})` |
| **Servidor MCP local** | `FastMCP` + `@mcp.tool()` + `mcp.run(transport="streamable-http")` |
| **Múltiplos servidores** | `MultiServerMCPClient(config_multi).get_tools()` |
| **Agente com MCP** | `create_react_agent(llm, tools=ferramentas_mcp)` |
| **MCP Registry** | `GET /v0/servers` — descoberta programática via API REST |
| **Servidor público** | Basta trocar a URL — protocolo idêntico para local e remoto |



### Links de referência:
| Recurso | URL |
|---|---|
| MCP Registry Oficial | https://registry.modelcontextprotocol.io/ |
| LangChain MCP Adapters | https://github.com/langchain-ai/langchain-mcp-adapters |
| MCP Spec (Anthropic) | https://modelcontextprotocol.io/ |
| LangGraph | https://langchain-ai.github.io/langgraph/ |


In [38]:
# ============================================================
# LIMPEZA – Encerra o servidor MCP local
# ============================================================
# Sempre encerre o servidor ao terminar para liberar a porta.

try:
    proc_servidor.terminate()
    proc_servidor.wait(timeout=5)
    print(f"✅ Servidor MCP local encerrado (PID {proc_servidor.pid})")
except Exception as e:
    print(f"ℹ️  Servidor já encerrado ou não iniciado: {e}")


✅ Servidor MCP local encerrado (PID 10008)
